# 08 — Multi-protein training: did the model learn to fold?

This is the generalization experiment after the six architecture notebooks and single-protein training. Train on several proteins, hold others out, and test whether the model learned an MSA-to-structure mapping instead of memorizing one fold.

In [ ]:
import sys
import re
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")
from af2_from_scratch import AF2Config

plt.rcParams["figure.figsize"] = (10, 4)
dev = "cuda" if torch.cuda.is_available() else "cpu"
VAL = set(
    open("../configs/splits/val.txt").read().split()
)  # strict + homolog val, from configs/splits/val.txt

## 1. Training curves
Train RMSD per protein (thin lines) vs **held-out val RMSD** (thick lines). Log scale — every halving matters.

In [ ]:
def parse_log(path="../logs/train_multi.log"):
    train, val = {}, {}
    p_step = re.compile(r"step\s+(\d+) \| (\w+)\s+\|.*RMSD\s+([\d.]+)")
    p_val = re.compile(r"(\w+):([\d.]+)")
    step_now = 0
    for line in open(path):
        m = p_step.search(line)
        if m:
            step_now = int(m.group(1))
            train.setdefault(m.group(2), []).append((step_now, float(m.group(3))))
        if ">> VAL" in line:
            for name, rmsd in p_val.findall(line):
                val.setdefault(name, []).append((step_now, float(rmsd)))
    return train, val


train, val = parse_log()
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for name, pts in train.items():
    s, r = zip(*pts)
    ax[0].plot(s, r, alpha=0.6, label=name)
for name, pts in val.items():
    s, r = zip(*pts)
    ax[0].plot(s, r, lw=3, label=f"VAL {name}")
ax[0].set_yscale("log")
ax[0].set_title("CA-RMSD vs teacher (A)")
ax[0].legend(fontsize=8)
ax[0].set_xlabel("step")
if val:
    last = {n: p[-1][1] for n, p in val.items()}
    ax[1].bar(last.keys(), last.values(), color="tomato")
    ax[1].set_title("latest VAL RMSD")
    ax[1].axhline(2, ls="--", c="gray")
    ax[1].set_ylabel("A")
plt.tight_layout()
plt.show()

## 2. Full evaluation of the latest checkpoint
Every protein, recycles=3 (AF-style inference). **Val bars are red.**

In [ ]:
from af2_from_scratch.dataset import ProteinDataset
from af2_from_scratch import AlphaFold2FromScratch
from af2_from_scratch.feature_extraction import sample_batch
from af2_from_scratch.geometry import kabsch_rmsd

ck = torch.load("../checkpoints/multi.pt", map_location=dev, weights_only=False)
cfg = AF2Config(**ck["cfg"])
step = ck.get("step", "?")
model = AlphaFold2FromScratch(cfg).to(dev)
model.load_state_dict(ck["model"])
model.eval()
print(f"checkpoint from step {step}")
data = ProteinDataset("../data")
tgts = {n: {k: v.to(dev) for k, v in data.targets[n].items()} for n in data.names}


@torch.no_grad()
def evaluate(n_clu=None, mask_p=0.0, query_only=False):
    scores = {}
    for n in data.names:
        k = 1 if query_only else (n_clu or cfg.n_clu)
        b = {
            kk: v.to(dev)
            for kk, v in sample_batch(
                data.features[n], k, k, mask_p=mask_p, seed=42
            ).items()
        }
        scores[n] = kabsch_rmsd(model(b, recycles=3)["ca"], tgts[n]["CA"]).item()
    return scores


full = evaluate()
names = data.names
colors = ["tomato" if n in VAL else "steelblue" for n in names]
plt.bar(names, [full[n] for n in names], color=colors)
plt.ylabel("CA-RMSD (A)")
plt.title("full MSA, recycles=3 — red = held-out")
plt.xticks(rotation=45)
plt.show()
print({n: round(v, 2) for n, v in full.items()})

## 3. THE memorization detector: query-only ablation
Single-protein training taught us: a model that memorized can fold **without any MSA** (0.08 A!).\nIf multi-protein training generalized, the picture should flip:\n* train proteins: query-only still works somewhat (partially memorized)\n* **val proteins: query-only MUST fail** — there is nothing to memorize, the MSA has to do the work

In [ ]:
qonly = evaluate(query_only=True)
x = range(len(names))
w = 0.35
plt.bar(
    [i - w / 2 for i in x],
    [full[n] for n in names],
    w,
    label="full MSA",
    color="steelblue",
)
plt.bar(
    [i + w / 2 for i in x],
    [qonly[n] for n in names],
    w,
    label="query only",
    color="orange",
)
plt.xticks(list(x), names, rotation=45)
plt.ylabel("CA-RMSD (A)")
for i, n in enumerate(names):
    if n in VAL:
        plt.axvspan(i - 0.5, i + 0.5, color="tomato", alpha=0.15)
plt.legend()
plt.title("red shading = held-out: query-only should collapse there")
plt.show()
print("query-only:", {n: round(v, 2) for n, v in qonly.items()})

## 4. See the held-out folds
Teacher vs student for crambin and hbb — proteins the model **never trained on**.

In [ ]:
def align(p, q):
    p, q = p - p.mean(0), q - q.mean(0)
    U, _, Vh = torch.linalg.svd(p.T @ q)
    d = torch.sign(torch.linalg.det(U @ Vh))
    return p @ (U @ torch.diag(torch.tensor([1.0, 1.0, d], device=p.device)) @ Vh), q


@torch.no_grad()
def fold(name):
    b = {
        k: v.to(dev)
        for k, v in sample_batch(
            data.features[name], cfg.n_clu, cfg.n_ext, mask_p=0.0, seed=42
        ).items()
    }
    out = model(b, recycles=3)
    ca_p, ca_t = (x.cpu() for x in align(out["ca"], tgts[name]["CA"]))
    return ca_p, ca_t


fig = plt.figure(figsize=(13, 5))
for i, name in enumerate(sorted(VAL)):
    ca_p, ca_t = fold(name)
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.plot(*ca_t.numpy().T, "o-", ms=4, lw=1.2, label="teacher (AF DB)")
    ax.plot(*ca_p.numpy().T, "s-", ms=3, lw=1.2, label="student")
    ax.set_title(f"{name} (HELD OUT) — RMSD {(ca_p - ca_t).norm(dim=-1).mean():.2f} A")
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Why is a protein hard? MSA depth vs error
The shallow-MSA proteins (crambin ~317, blg ~355 seqs) are the interesting cases — co-evolution signal is scarce.

In [ ]:
depth = {n: data.features[n]["msa_aatype"].shape[0] for n in names}
size = {n: data.features[n]["msa_aatype"].shape[1] for n in names}
plt.scatter(
    [depth[n] for n in names],
    [full[n] for n in names],
    s=[size[n] for n in names],
    c=["tomato" if n in VAL else "steelblue" for n in names],
)
for n in names:
    plt.annotate(
        n, (depth[n], full[n]), fontsize=8, xytext=(5, 3), textcoords="offset points"
    )
plt.xscale("log")
plt.xlabel("MSA depth (unique seqs, log)")
plt.ylabel("CA-RMSD (A)")
plt.title("marker size = protein length; red = held-out")
plt.show()

## 6. Interpretation guide
* **val RMSD < ~4-6 A**: right fold on unseen proteins — the architecture learned the mapping. Celebrate.
* **val RMSD < 2 A**: approaching AF-quality on these small proteins.
* **val stuck high while train -> 0**: memorization persists -> more proteins, fewer params, or more masking.
* **query-only collapses on val but not train**: exactly what we want to see — both effects coexist.

## Next experiments
1. **Scale proteins**: 20-50 more UniProt entries (rerun `fetch_data.py` with a bigger list)
2. **Scale model**: `n_evo=8, c_z=128` in a second overnight run — does capacity or data limit us?
3. **MSA-depth ablation**: train with `n_clu=32` — how much evolution is enough?
4. Compare student pLDDT vs teacher pLDDT per residue on hbb (calibration on an unseen protein)

In [ ]:
# your analysis here